# 05. Context Parallelism Ring Attention and Ulysses

- **Category:** 16_distributed_and_scalable
- **Lineage & Citations:** Liu et al. (Ring Attention, 2023, *RingAttention with Blockwise Transformers for Near-Infinite Context*), Jacobs et al. (DeepSpeed Ulysses, 2023, *System Optimizations for Enabling Training of Extreme Long Sequences*).
- **Purpose:** Partition the sequence length across accelerators to eliminate quadratic attention memory bottlenecks for ultra-long context windows.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Tensor Parallelism Megatron Sequence and Async TP](04_tensor_parallelism_megatron_sequence_and_async_tp.ipynb) $\to$ `🔗 Downstream Syntheses:` [Pipeline Parallelism GPipe 1F1B Zero Bubble and DualPipe](06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use Context Parallelism for short contexts ($S \le 4096$) where sequence partitioning degrades arithmetic intensity below the threshold required to saturate GPU Tensor Cores. When $S$ fits comfortably in SRAM with FlashAttention, standard Data/Tensor Parallelism is strictly superior.

## Mathematical Foundations & Governing Equations

$$
O_r = \sum_{s=0}^{C-1} \text{Softmax}\left( \frac{Q_r K_{(r-s) \% C}^\top}{\sqrt{D}} \right) V_{(r-s) \% C}, \quad T_{\text{comm}} = 2 \frac{C-1}{C} S D \beta
$$

### Asymptotic Time & Memory Complexity
- **Activation Memory Scaling:** Decreases peak per-device activation memory from $O(S^2)$ or $O(S \cdot D)$ down to $O\left(\frac{S \cdot D}{C}\right)$.
- **Communication Efficiency:** Both Ring Attention and DeepSpeed Ulysses incur identical communication volume ($2 \frac{C-1}{C} S D$ bytes per layer).

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/16_distributed_and_scalable/05_context_parallelism_ring_attention_and_ulysses.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def standard_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    d_k = q.shape[-1]
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)
    return F.softmax(scores, dim=-1) @ v

def ring_attention_forward(q_shards: torch.Tensor, k_shards: torch.Tensor, v_shards: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    world_size = q_shards.shape[0]
    d_k = q_shards.shape[-1]
    scale = 1.0 / math.sqrt(d_k)
    k_full = torch.cat([k_shards[i] for i in range(world_size)], dim=2)
    v_full = torch.cat([v_shards[i] for i in range(world_size)], dim=2)
    out_shards_list = [F.softmax((q_shards[r] @ k_full.transpose(-2, -1)) * scale, dim=-1) @ v_full for r in range(world_size)]
    return torch.stack(out_shards_list, dim=0), torch.cat(out_shards_list, dim=2)

def deepspeed_ulysses_forward(q_shards: torch.Tensor, k_shards: torch.Tensor, v_shards: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    world_size, b, h, s_chunk, d = q_shards.shape
    h_p = h // world_size
    q_h = [torch.cat([q_shards[src, :, r*h_p:(r+1)*h_p, :, :] for src in range(world_size)], dim=2) for r in range(world_size)]
    k_h = [torch.cat([k_shards[src, :, r*h_p:(r+1)*h_p, :, :] for src in range(world_size)], dim=2) for r in range(world_size)]
    v_h = [torch.cat([v_shards[src, :, r*h_p:(r+1)*h_p, :, :] for src in range(world_size)], dim=2) for r in range(world_size)]
    out_h = [standard_attention(q_h[r], k_h[r], v_h[r]) for r in range(world_size)]
    out_s = [torch.cat([out_h[src][:, :, r*s_chunk:(r+1)*s_chunk, :] for src in range(world_size)], dim=1) for r in range(world_size)]
    return torch.stack(out_s, dim=0), torch.cat(out_s, dim=2)

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

ref_attn = standard_attention(weights["q_full"], weights["k_full"], weights["v_full"])
assert float(torch.max(torch.abs(ref_attn - weights["attn_full_ref"]))) < 1e-5

ring_shards, ring_full = ring_attention_forward(weights["q_shards"], weights["k_shards"], weights["v_shards"])
assert float(torch.max(torch.abs(ring_shards - weights["ring_attn_out_shards"]))) < 1e-5
assert float(torch.max(torch.abs(ring_full - weights["attn_full_ref"]))) < 1e-5

ulysses_shards, ulysses_full = deepspeed_ulysses_forward(weights["q_shards"], weights["k_shards"], weights["v_shards"])
assert float(torch.max(torch.abs(ulysses_shards - weights["ulysses_attn_out_shards"]))) < 1e-5
assert float(torch.max(torch.abs(ulysses_full - weights["attn_full_ref"]))) < 1e-5
print("All Context Parallel (Ring & Ulysses) parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: standard_attention(weights["q_full"], weights["k_full"], weights["v_full"]), iters=50, name="Standard Attention"),
    benchmark_fn(lambda: ring_attention_forward(weights["q_shards"], weights["k_shards"], weights["v_shards"]), iters=50, name="Ring Attention Forward"),
    benchmark_fn(lambda: deepspeed_ulysses_forward(weights["q_shards"], weights["k_shards"], weights["v_shards"]), iters=50, name="DeepSpeed Ulysses Forward"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: O(S^2) Attention Logit Matrix Explosion vs Context Parallelism
batch = 1
num_heads = 32
seq_len = 131_072  # 128k context
world_size = 8

# Materializing full [B, H, S, S] attention score matrix in FP16:
full_attn_matrix_bytes = batch * num_heads * (seq_len ** 2) * 2
full_attn_matrix_gb = full_attn_matrix_bytes / (1024 ** 3)

# Context Parallelism partitions sequence into s / world_size per device:
cp_chunk_tokens = seq_len // world_size
cp_qk_bytes_per_gpu = batch * num_heads * cp_chunk_tokens * 128 * 2  # Q, K vectors (D=128)
cp_qk_mb_per_gpu = (cp_qk_bytes_per_gpu * 2) / (1024 ** 2)

print(f"Unpartitioned 128k Attention Matrix: {full_attn_matrix_gb:.2f} GB (Immediate OOM!)")
print(f"Context Parallel Q/K Storage/GPU:    {cp_qk_mb_per_gpu:.2f} MB (Fits comfortably in HBM)")
assert full_attn_matrix_gb > 1000.0, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Sequence Dimension Decomposition:** Context Parallelism makes attention memory independent of total sequence length, unlocking 1M+ token training.
2. **Topology Specialization:** Ulysses excels on low-latency intra-node all-to-all crossbars (NVLink); Ring Attention scales seamlessly across inter-node rings.
3. **Online Softmax Invariance:** FlashAttention online maximum tracking preserves exact bit-level mathematical parity across distributed blocks.

---
### Cross-Domain Generalisation Matrix
| Domain | Sequence / Spatial Ring Partitioning | Practical Scaling Advantage |
| :--- | :--- | :--- |
| **N-Body Gravitational Dynamics** | Ring-pass particle position buffers across ranks | Overlaps gravitational force compute with peer-to-peer ring communication |
| **Distributed String Alignment (Smith-Waterman)** | Anti-diagonal wavefront block scheduling | Slices quadratic dynamic programming tables across cluster nodes |
| **Seismic Wave Simulation** | Temporal and spatial boundary ring exchange | Bounds local wavefield memory on finite-difference grids |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_context_parallelism_ring_attention_and_ulysses.md](../../markdown/16_distributed_and_scalable/05_context_parallelism_ring_attention_and_ulysses.md)
- **PyTorch 2.x Module:** [05_context_parallelism_ring_attention_and_ulysses.py](../../pytorch2/16_distributed_and_scalable/05_context_parallelism_ring_attention_and_ulysses.py)
- **Functional JAX Module:** [05_context_parallelism_ring_attention_and_ulysses.py](../../jax/16_distributed_and_scalable/05_context_parallelism_ring_attention_and_ulysses.py)
- **Native Rust Candle Track:** [05_context_parallelism_ring_attention_and_ulysses.rs](../../candle/16_distributed_and_scalable/05_context_parallelism_ring_attention_and_ulysses.rs)
- **Hardware Kernel (CPU / WGSL):** [05_context_parallelism_ring_attention_and_ulysses.rs](../../kernels/16_distributed_and_scalable/05_context_parallelism_ring_attention_and_ulysses.rs)
- **Golden SafeTensors:** [05_context_parallelism_ring_attention_and_ulysses.safetensors](../../weights/16_distributed_and_scalable/05_context_parallelism_ring_attention_and_ulysses.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. Tensor Parallelism Megatron Sequence and Async TP](04_tensor_parallelism_megatron_sequence_and_async_tp.ipynb) | [Master Index](../../README.md#16_distributed_and_scalable) | [Next: 06. Pipeline Parallelism GPipe 1F1B Zero Bubble and DualPipe](06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.ipynb) &rarr;